# Vesuvius Surface Detection -- skeleton-recall (700ep) + 1st-place postprocessing submission

Runs inference with the skeleton-recall winner-extension checkpoint (from-scratch training,
DC+CE+skeleton-recall loss, LOSO fold_0, scroll 26010 held out), then applies the 1st-place
control postprocessing chain (`src/postprocess/first_place.py`: remove_small -> closing ->
height-map patch -> hole plug -> fill_holes) before packaging `submission.zip`.

Adapted from the proven `vesuvius-1000-epoch-loso-submission` kernel (same mount/offline-install/
staging mechanics, unchanged) -- only the checkpoint, trainer name, and the new postprocessing
step differ. See `baselinerun/research_log.md` and `vesuvius-surface/research_log.md` sections
6/14 for the skeleton-recall validation story and the postprocessing control's own ablation.


## Configuration

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path
from typing import Optional, Tuple, Union

# ---- Kaggle dataset slugs -- UPDATE CHECKPOINT_DATASET_SLUG once the real 700-epoch checkpoint
# is uploaded. Currently pointed at a mid-training pipeline-test checkpoint (~epoch 200) to
# prove the full mechanics (staging, inference, NEW postprocessing step, submission packaging)
# before committing to the real run's kernel push -- same pattern already used for the
# 1000-epoch model's own development (see submissions 55303075, 55311026). ----
CHECKPOINT_DATASET_SLUG = "vesuvius-skelrecall700-checkpoint-v1"  # REAL final checkpoint (epoch 700, training complete)
WHEELS_DATASET_SLUG = "vesuvius-nnunet-wheels-v3"

# ---- Paths (Kaggle mounts) ----
INPUT_DIR = Path("/kaggle/input/competitions/vesuvius-challenge-surface-detection")
KAGGLE_USERNAME = "vigneshk96"
# Kaggle's /kaggle/input mount convention for custom (dataset_sources) datasets is NOT
# consistent -- confirmed directly via two throwaway diagnostic kernel runs
# (vigneshk96/vesuvius-diag-mount) that ran `mount` + `ls -R /kaggle/input`. Run 1: both
# attached datasets mounted at the NEW nested path /kaggle/input/datasets/<owner>/<slug>/.
# Run 2 (different dataset, same kernel/account): the wheels dataset again mounted at the
# nested path, but the checkpoint dataset mounted at the OLD flat /kaggle/input/<slug>/
# instead -- two datasets, two different conventions, same run. Real, observed platform
# behavior, not something to assume away. CHECKPOINT_DIR/WHEELS_DIR are resolved for real
# below (see resolve_dataset_mount in the offline-install cell) by probing both candidate
# paths and using whichever one actually exists -- these are just the candidate list.
CHECKPOINT_DIR_CANDIDATES = [
    Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{CHECKPOINT_DATASET_SLUG}"),
    Path(f"/kaggle/input/{CHECKPOINT_DATASET_SLUG}"),
]
WHEELS_DIR_CANDIDATES = [
    Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{WHEELS_DATASET_SLUG}"),
    Path(f"/kaggle/input/{WHEELS_DATASET_SLUG}"),
]
CHECKPOINT_DIR = CHECKPOINT_DIR_CANDIDATES[0]  # placeholder, re-resolved for real below
WHEELS_DIR = WHEELS_DIR_CANDIDATES[0]  # placeholder, re-resolved for real below
WORKING_DIR = Path("/kaggle/temp")
OUTPUT_DIR = Path("/kaggle/working")

NNUNET_RAW = WORKING_DIR / "nnUNet_data" / "nnUNet_raw"
NNUNET_PREPROCESSED = WORKING_DIR / "nnUNet_data" / "nnUNet_preprocessed"
NNUNET_RESULTS = WORKING_DIR / "nnUNet_results"

# ---- Model identity (must match what was trained) ----
DATASET_ID = 100
DATASET_NAME = "Dataset100_VesuviusSurface"
CONFIGURATION = "3d_lowres"
PLANS_NAME = "nnUNetResEncUNetMPlans"
TRAINER = "nnUNetTrainerSkeletonRecall_700epochs"  # stub trainer staged below -- inference only
                                                     # needs build_network_architecture, which
                                                     # skeleton-recall never overrides, so the
                                                     # full loss/transform code isn't needed here
                                                     # (verified: nnUNetv2_predict calls
                                                     # build_network_architecture directly on the
                                                     # class without instantiating it, __init__
                                                     # never runs at inference time)
CHECKPOINT_FILENAME = "checkpoint_best.pth"
FOLD = 0  # scroll-grouped split: fold 0 = scroll 26010 held out (see splits_final.json)

MODEL_DIR_NAME = f"{TRAINER}__{PLANS_NAME}__{CONFIGURATION}"
EXT_TRAINER_DIR = WORKING_DIR / "ext_trainers"

TEST_INPUT_DIR = WORKING_DIR / "test_input"
PREDICTIONS_DIR = WORKING_DIR / "predictions"
PREDICTIONS_TIFF_DIR = OUTPUT_DIR / "predictions_tiff"
POSTPROCESSED_DIR = OUTPUT_DIR / "predictions_postprocessed"
SUBMISSION_ZIP = OUTPUT_DIR / "submission.zip"

print("INPUT_DIR:", INPUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("WHEELS_DIR:", WHEELS_DIR)


## Offline install

Real internet is disabled during Kaggle submission scoring. Installs `nnunetv2==2.8.1`
(exactly the version the checkpoint was trained with -- avoids any train/inference version
drift) plus its non-torch dependencies from the offline wheel bundle. torch/torchvision are
deliberately *not* bundled: Kaggle's GPU notebook image already ships a working torch, and
pulling a mismatched one from PyPI's default index during wheel-building actually resolved to
the wrong CUDA toolkit version entirely -- see `baselinerun/kaggle_submission/README.md`.

The wheel bundle is built for **Python 3.12** (`cp312`) -- Kaggle's actual kernel Python,
confirmed from a real failed run's traceback paths (`/usr/local/lib/python3.12/...`), not
assumed. The first build targeted 3.11 to match the local dev env and failed on Kaggle with
`ERROR: Could not find a version that satisfies the requirement nnunetv2==2.8.1 (from
versions: none)` -- pip correctly refused to install `cp311`-tagged compiled wheels
(numpy/scipy/scikit-image/etc.) into a 3.12 interpreter.

In [ ]:
import time


def resolve_dataset_mount(candidates: list, min_files: int = 1, timeout_s: int = 1800, poll_s: int = 15) -> Path:
    """Polls every candidate path each round (Kaggle's mount convention for custom datasets
    is inconsistent even within one account/kernel run -- see the cell-2 comment) and returns
    the first one that actually has content. Raises only if NONE of the candidates mount
    within timeout_s."""
    waited = 0
    while waited <= timeout_s:
        for c in candidates:
            if c.exists() and len(list(c.iterdir())) >= min_files:
                print(f"{c} mounted ({len(list(c.iterdir()))} entries) after {waited}s")
                return c
        time.sleep(poll_s)
        waited += poll_s
    raise RuntimeError(
        f"none of {candidates} mounted after {timeout_s}s "
        f"(exists={[c.exists() for c in candidates]})"
    )


def wait_for_mount(path: Path, min_files: int = 1, timeout_s: int = 900, poll_s: int = 15) -> None:
    """Single-path convenience wrapper around resolve_dataset_mount, for mounts whose path
    convention is already known/stable (e.g. competition data, which has consistently used
    the /kaggle/input/competitions/<slug>/ nesting in every run so far -- unlike the custom
    dataset_sources paths, which need the multi-candidate resolver above)."""
    resolve_dataset_mount([path], min_files=min_files, timeout_s=timeout_s, poll_s=poll_s)


WHEELS_DIR = resolve_dataset_mount(WHEELS_DIR_CANDIDATES, min_files=50, timeout_s=1800)
CHECKPOINT_DIR = resolve_dataset_mount(CHECKPOINT_DIR_CANDIDATES, min_files=1, timeout_s=1800)

result = subprocess.run(
    f"pip install --no-index --find-links={WHEELS_DIR} nnunetv2==2.8.1 nibabel tifffile tqdm -q",
    shell=True, capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("Offline install failed")

import nnunetv2
print("nnunetv2 installed OK")


## Environment setup

Verbatim from `baselinerun/src/training/environment.py::setup_environment`.

In [ ]:
def setup_environment():
    for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, OUTPUT_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    os.environ["nnUNet_raw"] = str(NNUNET_RAW)
    os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
    os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
    # baselinerun's own config uses "true" (torch.compile), but Kaggle's GPU pool can assign
    # older cards (confirmed: Tesla P100, CUDA capability 6.0) that torch.compile's Triton
    # backend does not support (requires >=7.0) -- a real run hit
    # "torch._inductor.exc.GPUTooOldForTriton" here. Disabled for Kaggle specifically;
    # arunodhayan's own notebook independently made the same call for the same reason.
    os.environ["nnUNet_compile"] = "false"

    print(f"nnUNet_raw: {NNUNET_RAW}")
    print(f"nnUNet_preprocessed: {NNUNET_PREPROCESSED}")
    print(f"nnUNet_results: {NNUNET_RESULTS}")
    print(f"nnUNet_USE_BLOSC2: {os.environ.get('nnUNet_USE_BLOSC2', 'not set')} (0=NPZ, 1=blosc2)")


setup_environment()


## Stage the checkpoint and the custom trainer

Copies the uploaded checkpoint dataset into the layout `nnUNetv2_predict` expects:
`nnUNet_results/{DATASET_NAME}/{Trainer}__{Plans}__{Config}/{dataset.json,plans.json,fold_0/checkpoint_best.pth}`.

The checkpoint dataset is uploaded *flat* (dataset.json, plans.json, checkpoint_best.pth all
at the dataset root, no subfolder) deliberately -- an earlier version uploaded a nested folder
via `kaggle datasets create -r zip`, which bundles the whole folder as a single `.zip` file
rather than unpacking it, and that dataset consistently failed to mount into a running kernel
at all (confirmed: still not mounted after a 15-minute wait, while flat, non-zipped datasets
mounted instantly). Flat upload avoids the zip path entirely.

Also writes a minimal stub `nnUNetTrainerSkeletonRecall_700epochs.py` to disk and sets
`nnUNet_extTrainer` so `nnUNetv2_predict` can resolve the trainer by name (the checkpoint
embeds this trainer name internally). The real trainer adds a skeleton-recall loss + training
transforms that never run at inference -- see the code cell below for why a bare stub is
sufficient (verified from nnU-Net's own inference source, not assumed).

In [ ]:
dst_model_dir = NNUNET_RESULTS / DATASET_NAME / MODEL_DIR_NAME
dst_model_dir.mkdir(parents=True, exist_ok=True)
(dst_model_dir / f"fold_{FOLD}").mkdir(parents=True, exist_ok=True)

shutil.copy2(CHECKPOINT_DIR / "dataset.json", dst_model_dir / "dataset.json")
shutil.copy2(CHECKPOINT_DIR / "plans.json", dst_model_dir / "plans.json")
shutil.copy2(CHECKPOINT_DIR / CHECKPOINT_FILENAME, dst_model_dir / f"fold_{FOLD}" / CHECKPOINT_FILENAME)

checkpoint_path = dst_model_dir / f"fold_{FOLD}" / CHECKPOINT_FILENAME
assert checkpoint_path.exists(), f"Missing checkpoint at {checkpoint_path}"
print("Staged checkpoint:", checkpoint_path, f"({checkpoint_path.stat().st_size / 1e6:.1f} MB)")
print("dataset.json:", (dst_model_dir / "dataset.json").exists())
print("plans.json:", (dst_model_dir / "plans.json").exists())

# ---- Stage a minimal stub trainer so nnUNetv2_predict can resolve
# nnUNetTrainerSkeletonRecall_700epochs by name via the external-trainer mechanism.
# The real trainer (vesuvius-surface/src/training/trainers/nnUNetTrainerSkeletonRecall*.py)
# adds a skeleton-recall loss term and extra training-time transforms -- none of that runs at
# inference. nnUNetv2_predict resolves the trainer class from the checkpoint's embedded
# trainer_name and calls build_network_architecture DIRECTLY ON THE CLASS, never instantiating
# it (verified via source read of predict_from_raw_data.py -- no trainer_class(...) call
# anywhere in the inference path). Confirmed neither nnUNetTrainerSkeletonRecall nor
# nnUNetTrainerSkeletonRecall_700epochs override build_network_architecture (source-read) --
# it's inherited unchanged from stock nnUNetTrainer, so a bare passthrough subclass is
# functionally identical for inference purposes. ----
EXT_TRAINER_DIR.mkdir(parents=True, exist_ok=True)
(EXT_TRAINER_DIR / "nnUNetTrainerSkeletonRecall_700epochs.py").write_text('''\
"""Inference-only stub for nnUNetTrainerSkeletonRecall_700epochs.

The real trainer (vesuvius-surface/src/training/trainers/nnUNetTrainerSkeletonRecall.py and
its _700epochs subclass) adds a DC+CE+skeleton-recall loss and extra training-time transforms
on top of stock nnUNetTrainer. Neither overrides build_network_architecture, and
nnUNetv2_predict calls build_network_architecture directly on the resolved class without ever
instantiating it -- so at inference time this bare subclass is functionally identical to the
real one. Loaded via nnU-Net's external-trainer mechanism (env var nnUNet_extTrainer); see
nnunetv2.utilities.find_objects.recursive_find_trainer_class_by_name.
"""

from __future__ import annotations

from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainerSkeletonRecall_700epochs(nnUNetTrainer):
    pass
''')
os.environ["nnUNet_extTrainer"] = str(EXT_TRAINER_DIR)
print("Staged custom trainer at:", EXT_TRAINER_DIR / "nnUNetTrainerSkeletonRecall_700epochs.py")
print("nnUNet_extTrainer:", os.environ["nnUNet_extTrainer"])

## Prepare test data

Verbatim from `baselinerun/src/data/prepare_training_data.py`
(`create_spacing_json`, `prepare_single_case`, `prepare_test_data`) -- only the test-data
subset of that module is needed here, since this notebook only runs inference.

In [ ]:
from tqdm.auto import tqdm


def create_spacing_json(output_path: Path, shape: tuple, spacing: tuple = (1.0, 1.0, 1.0)):
    json_data = {"spacing": list(spacing)}
    with open(output_path, "w") as f:
        json.dump(json_data, f)


def prepare_single_case(src_path: Path, dest_path: Path, json_path: Path, use_symlinks: bool = True) -> bool:
    try:
        import tifffile
        with tifffile.TiffFile(src_path) as tif:
            shape = tif.pages[0].shape if len(tif.pages) == 1 else (len(tif.pages), *tif.pages[0].shape)

        if use_symlinks:
            if not dest_path.exists():
                dest_path.symlink_to(src_path.resolve())
        else:
            shutil.copy2(src_path, dest_path)

        create_spacing_json(json_path, shape)
        return True
    except Exception as e:
        print(f"Error processing {src_path.name}: {e}")
        return False


def prepare_test_data(input_dir: Path, output_dir: Path, use_symlinks: bool = True) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    test_images_dir = input_dir / "test_images"
    if not test_images_dir.exists():
        raise FileNotFoundError(f"{test_images_dir} not found")

    test_files = sorted(test_images_dir.glob("*.tif"))
    print(f"Found {len(test_files)} test cases")

    for img_path in tqdm(test_files, desc="Preparing test data"):
        case_id = img_path.stem
        prepare_single_case(img_path, output_dir / f"{case_id}_0000.tif", output_dir / f"{case_id}_0000.json", use_symlinks)

    return output_dir


# The competition dataset (attached via competition_sources, not dataset_sources) previously
# hit the exact same mount-lag issue as our own datasets -- confirmed via the Kaggle API that
# test_images/1407735.tif genuinely exists, yet a run still saw test_images/ never mount even
# after a full 15-minute wait (unlike our own datasets, which always mounted instantly once
# uploaded correctly). Diagnose broadly this time instead of assuming the fix is identical:
# wait for INPUT_DIR itself first, print what's actually there, then handle test_images
# specifically with clear diagnostics either way.
wait_for_mount(INPUT_DIR, min_files=1, timeout_s=300)
print("INPUT_DIR contents:", sorted(p.name for p in INPUT_DIR.iterdir()))

test_images_dir = INPUT_DIR / "test_images"
try:
    wait_for_mount(test_images_dir, min_files=1, timeout_s=600)
except RuntimeError as e:
    print(f"WARNING: {e}")
    print("Trying case-insensitive / alternate-name search under INPUT_DIR...")
    candidates = [p for p in INPUT_DIR.rglob("*") if p.is_dir() and "test" in p.name.lower()]
    print("Directories with 'test' in the name:", candidates)
    for c in candidates:
        try:
            print(f"  {c}: {sorted(p.name for p in c.iterdir())[:10]}")
        except Exception as inner_e:
            print(f"  {c}: could not list ({inner_e})")
    raise

prepare_test_data(INPUT_DIR, TEST_INPUT_DIR)


## Run inference

`_run_command` and `run_inference` verbatim from `baselinerun/src/training/commands.py`.

In [ ]:
def _run_command(cmd: str, name: str = "Command", tail_lines: int = 30, timeout: Optional[int] = None) -> bool:
    print(f"Running: {cmd}")
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"{name} TIMEOUT after {timeout}s!")
        return False

    if result.returncode != 0:
        print(f"{name} FAILED!")
        print(f"STDERR:\n{result.stderr[-3000:]}")
        return False

    print(f"{name} complete!")
    if result.stdout.strip():
        lines = result.stdout.strip().split("\n")
        print("\n".join(lines[-tail_lines:]))
    return True


def run_inference(
    input_dir: Path, output_dir: Path, dataset_id: int, config: str, fold: Union[int, str],
    plans: str, trainer: str, checkpoint_name: str = "checkpoint_final.pth", save_probabilities: bool = True,
    num_processes_preprocessing: int = 2, num_processes_segmentation: int = 2,
) -> bool:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = f"nnUNetv2_predict -d {dataset_id:03d} -c {config} -f {fold}"
    cmd += f" -i {input_dir} -o {output_dir} -p {plans} -tr {trainer} -chk {checkpoint_name}"
    cmd += f" -npp {num_processes_preprocessing} -nps {num_processes_segmentation}"
    cmd += " --verbose"
    if save_probabilities:
        cmd += " --save_probabilities"
    return _run_command(cmd, "Inference")


ok = run_inference(
    input_dir=TEST_INPUT_DIR, output_dir=PREDICTIONS_DIR,
    dataset_id=DATASET_ID, config=CONFIGURATION, fold=FOLD,
    plans=PLANS_NAME, trainer=TRAINER, checkpoint_name=CHECKPOINT_FILENAME,
)
assert ok, "Inference failed -- see STDERR above"


## Convert predictions to submission TIFFs

`load_probabilities` and `predictions_to_tiff` verbatim from
`baselinerun/src/training/postprocess.py` (the `.nii.gz` fallback branch is dropped here --
it exists in the source only for a legacy prediction path this notebook never produces).

In [ ]:
import numpy as np
import tifffile


def load_probabilities(npz_path: Path) -> np.ndarray:
    data = np.load(npz_path)
    return data["probabilities"]


def predictions_to_tiff(pred_dir: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    npz_files = list(pred_dir.glob("*.npz"))
    tif_files = list(pred_dir.glob("*.tif"))

    if npz_files:
        print(f"Converting {len(npz_files)} NPZ probability files to TIFF...")
        for npz_path in tqdm(npz_files, desc="Converting to TIFF"):
            case_id = npz_path.stem
            probs = load_probabilities(npz_path)
            pred = np.argmax(probs, axis=0).astype(np.uint8)
            tifffile.imwrite(output_dir / f"{case_id}.tif", pred)
    elif tif_files:
        print(f"Copying {len(tif_files)} TIFF prediction files...")
        for tif_path in tqdm(tif_files, desc="Copying TIFF"):
            case_id = tif_path.stem
            pred = tifffile.imread(str(tif_path)).astype(np.uint8)
            tifffile.imwrite(output_dir / f"{case_id}.tif", pred)
    else:
        print(f"WARNING: No prediction files found in {pred_dir}")


predictions_to_tiff(PREDICTIONS_DIR, PREDICTIONS_TIFF_DIR)


## Apply 1st-place postprocessing

Inlined verbatim from `vesuvius-surface/src/postprocess/first_place.py` (the validated
control chain -- not the unmerge novelty layer, which is still under calibration and not part
of this submission). Same defaults as the writeup / research_log.md: remove components <20k
voxels, per-sheet binary closing (radius 3, 26-connectivity), height-map gap patching (discard
if it increases hole count), 1-voxel hole plugging via 2x2x2 LUT, then a global
`binary_fill_holes`. Applied per predicted TIFF, `PREDICTIONS_TIFF_DIR` -> `POSTPROCESSED_DIR`.

In [ ]:
from scipy import ndimage
from scipy.ndimage import (
    binary_closing,
    binary_fill_holes,
    distance_transform_edt,
    find_objects,
    generate_binary_structure,
    label as cc_label,
)
from dataclasses import dataclass


@dataclass
class PostprocessConfig:
    min_component_size: int = 20_000
    closing_radius: int = 3
    connectivity: int = 26
    enable_closing: bool = True
    enable_patching: bool = True
    enable_hole_plugging: bool = True
    enable_fill_holes: bool = True
    surface_label: int = 1
    threshold: float = 0.5


def binarize_prediction(volume, *, threshold=0.5, surface_label=1):
    arr = np.asarray(volume)
    if arr.ndim == 4:
        if arr.shape[0] > surface_label:
            arr = arr[surface_label]
        else:
            arr = arr.argmax(axis=0)
    if np.issubdtype(arr.dtype, np.floating):
        return (arr >= threshold).astype(np.uint8)
    uniq = set(np.unique(arr).tolist())
    if uniq <= {0, 1}:
        return (arr > 0).astype(np.uint8)
    return (arr == surface_label).astype(np.uint8)


def _structure(connectivity):
    if connectivity == 6:
        return generate_binary_structure(3, 1)
    if connectivity == 18:
        return generate_binary_structure(3, 2)
    if connectivity == 26:
        return generate_binary_structure(3, 3)
    raise ValueError(f"connectivity must be 6, 18, or 26; got {connectivity}")


def make_ball_footprint(radius):
    zz, yy, xx = np.ogrid[-radius : radius + 1, -radius : radius + 1, -radius : radius + 1]
    return (zz**2 + yy**2 + xx**2) <= radius**2


def _pad_slices(sl, shape, pad):
    return tuple(
        slice(max(0, s.start - pad), min(dim, s.stop + pad)) for s, dim in zip(sl, shape)
    )


def remove_small_components(mask, min_size=20_000, connectivity=26):
    struct = _structure(connectivity)
    labeled, n = cc_label(mask.astype(np.uint8), structure=struct)
    if n == 0:
        return mask.astype(np.uint8)
    sizes = ndimage.sum(mask, labeled, range(1, n + 1))
    keep = np.zeros_like(mask, dtype=np.uint8)
    for i, size in enumerate(sizes, 1):
        if size >= min_size:
            keep[labeled == i] = 1
    return keep


def _count_internal_holes(mask):
    inv = 1 - mask.astype(np.uint8)
    labeled, n = cc_label(inv, structure=generate_binary_structure(3, 1))
    if n == 0:
        return 0
    border = np.zeros(n + 1, dtype=bool)
    border[labeled[0]] = True
    border[labeled[-1]] = True
    border[labeled[:, 0]] = True
    border[labeled[:, -1]] = True
    border[labeled[:, :, 0]] = True
    border[labeled[:, :, -1]] = True
    border[0] = True
    return int(n - border[1:].sum())


def height_map_patch_crop(crop):
    if crop.sum() == 0:
        return crop

    best_axis, best_area = 0, 0
    for axis in range(3):
        area = int(crop.max(axis=axis).sum())
        if area > best_area:
            best_area = area
            best_axis = axis

    crop_t = np.moveaxis(crop, best_axis, 0)
    depth, height, width = crop_t.shape
    depth_coords = np.arange(depth, dtype=np.float32).reshape(depth, 1, 1)
    valid_3d = crop_t.astype(bool)
    count_map = valid_3d.sum(axis=0)
    has_voxels = count_map > 0

    height_map = np.full((height, width), np.nan, dtype=np.float32)
    thick_map = np.zeros((height, width), dtype=np.float32)
    depth_sum = (depth_coords * valid_3d).sum(axis=0)
    height_map[has_voxels] = depth_sum[has_voxels] / count_map[has_voxels]
    thick_map[has_voxels] = count_map[has_voxels]

    filled_proj = binary_fill_holes(has_voxels)
    gap_mask = filled_proj & ~has_voxels
    if not gap_mask.any():
        return crop

    holes_before = _count_internal_holes(crop)

    fill_row_h = np.full((height, width), np.nan, dtype=np.float32)
    fill_row_t = np.full((height, width), np.nan, dtype=np.float32)
    for r in range(height):
        valid_cols = np.where(has_voxels[r])[0]
        gap_cols = np.where(gap_mask[r])[0]
        if len(valid_cols) >= 2 and len(gap_cols) > 0:
            fill_row_h[r, gap_cols] = np.interp(gap_cols, valid_cols, height_map[r, valid_cols])
            fill_row_t[r, gap_cols] = np.interp(gap_cols, valid_cols, thick_map[r, valid_cols])

    fill_col_h = np.full((height, width), np.nan, dtype=np.float32)
    fill_col_t = np.full((height, width), np.nan, dtype=np.float32)
    for c in range(width):
        valid_rows = np.where(has_voxels[:, c])[0]
        gap_rows = np.where(gap_mask[:, c])[0]
        if len(valid_rows) >= 2 and len(gap_rows) > 0:
            fill_col_h[gap_rows, c] = np.interp(gap_rows, valid_rows, height_map[valid_rows, c])
            fill_col_t[gap_rows, c] = np.interp(gap_rows, valid_rows, thick_map[valid_rows, c])

    not_valid = ~has_voxels
    row_dist = distance_transform_edt(not_valid, sampling=[1e6, 1])
    col_dist = distance_transform_edt(not_valid, sampling=[1, 1e6])

    gap_r, gap_c = np.where(gap_mask)
    hr, hc = fill_row_h[gap_r, gap_c], fill_col_h[gap_r, gap_c]
    tr, tc = fill_row_t[gap_r, gap_c], fill_col_t[gap_r, gap_c]
    dr = np.maximum(row_dist[gap_r, gap_c], 1e-6)
    dc = np.maximum(col_dist[gap_r, gap_c], 1e-6)

    valid_r, valid_c = ~np.isnan(hr), ~np.isnan(hc)
    both = valid_r & valid_c
    only_r, only_c = valid_r & ~valid_c, valid_c & ~valid_r
    wr = np.where(both, 1.0 / dr, 0.0)
    wc = np.where(both, 1.0 / dc, 0.0)
    w_total = np.maximum(wr + wc, 1e-12)

    h_avg = np.where(
        both,
        (np.nan_to_num(hr) * wr + np.nan_to_num(hc) * wc) / w_total,
        np.where(only_r, hr, np.where(only_c, hc, np.nan)),
    )
    t_avg = np.where(
        both,
        (np.nan_to_num(tr) * wr + np.nan_to_num(tc) * wc) / w_total,
        np.where(only_r, tr, np.where(only_c, tc, 0.0)),
    )

    patched_t = crop_t.copy()
    for idx in range(len(gap_r)):
        h_val = h_avg[idx]
        if np.isnan(h_val):
            continue
        r, c = int(gap_r[idx]), int(gap_c[idx])
        center = int(round(float(h_val)))
        half = max(0, int(round(float(t_avg[idx]) / 2)))
        z0 = max(0, center - half)
        z1 = min(depth - 1, center + half)
        patched_t[z0 : z1 + 1, r, c] = 1

    patched_3d = np.moveaxis(patched_t, 0, best_axis)
    if _count_internal_holes(patched_3d) > holes_before:
        return crop
    return patched_3d.astype(np.uint8)


_HOLE_PLUG_LUT = None


def _build_hole_plug_lut():
    face_diags = [
        (0, 3, 1, 2), (1, 2, 0, 3), (4, 7, 5, 6), (5, 6, 4, 7),
        (0, 5, 1, 4), (1, 4, 0, 5), (2, 7, 3, 6), (3, 6, 2, 7),
        (0, 6, 2, 4), (2, 4, 0, 6), (1, 7, 3, 5), (3, 5, 1, 7),
    ]
    lut = np.zeros(256, dtype=np.uint8)
    for pattern in range(256):
        add = 0
        for fa, fb, g1, g2 in face_diags:
            if (
                ((pattern >> fa) & 1)
                and ((pattern >> fb) & 1)
                and not ((pattern >> g1) & 1)
                and not ((pattern >> g2) & 1)
            ):
                add |= 1 << g1
        lut[pattern] = add
    return lut


def plug_holes_lut(mask, max_iterations=5):
    global _HOLE_PLUG_LUT
    if _HOLE_PLUG_LUT is None:
        _HOLE_PLUG_LUT = _build_hole_plug_lut()
    lut = _HOLE_PLUG_LUT
    result = mask.astype(np.uint8).copy()
    depth, height, width = result.shape
    if depth < 2 or height < 2 or width < 2:
        return result

    offsets = [(dz, dy, dx) for dz in range(2) for dy in range(2) for dx in range(2)]
    for _ in range(max_iterations):
        pattern = np.zeros((depth - 1, height - 1, width - 1), dtype=np.uint8)
        for bit, (dz, dy, dx) in enumerate(offsets):
            pattern |= result[dz : depth - 1 + dz, dy : height - 1 + dy, dx : width - 1 + dx] << bit
        additions = lut[pattern]
        if not additions.any():
            break
        for bit, (dz, dy, dx) in enumerate(offsets):
            add_bit = ((additions >> bit) & 1).astype(np.uint8)
            result[dz : depth - 1 + dz, dy : height - 1 + dy, dx : width - 1 + dx] |= add_bit
    return result


def apply_first_place(prediction, config=None, *, through_stage="fill"):
    stages = ("raw", "remove_small", "closing", "patch", "plug", "fill")
    cfg = config or PostprocessConfig()
    stop_at = stages.index(through_stage)

    mask = binarize_prediction(prediction, threshold=cfg.threshold, surface_label=cfg.surface_label)
    if stop_at == 0:
        return mask

    mask = remove_small_components(mask, cfg.min_component_size, cfg.connectivity)
    if stop_at == 1:
        return mask

    struct = _structure(cfg.connectivity)
    labeled, n = cc_label(mask, structure=struct)
    if n == 0:
        return mask

    slices = find_objects(labeled)
    footprint = make_ball_footprint(cfg.closing_radius) if cfg.closing_radius > 0 else None
    pad = cfg.closing_radius if cfg.enable_closing and cfg.closing_radius > 0 else 0
    result = np.zeros_like(mask, dtype=np.uint8)

    do_closing = cfg.enable_closing and stop_at >= 2
    do_patch = cfg.enable_patching and stop_at >= 3
    do_plug = cfg.enable_hole_plugging and stop_at >= 4

    for comp_id, sl in enumerate(slices, 1):
        if sl is None:
            continue
        padded_sl = _pad_slices(sl, mask.shape, pad)
        crop = (labeled[padded_sl] == comp_id).astype(np.uint8)

        if do_closing and footprint is not None:
            crop = binary_closing(crop, structure=footprint).astype(np.uint8)
        if do_patch:
            crop = height_map_patch_crop(crop)
        if do_plug:
            crop = plug_holes_lut(crop)

        result[padded_sl] |= crop

    if stop_at < 5 or not cfg.enable_fill_holes:
        return result
    return binary_fill_holes(result).astype(np.uint8)


POSTPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
pp_config = PostprocessConfig()
pred_tiffs = sorted(PREDICTIONS_TIFF_DIR.glob("*.tif"))
print(f"Postprocessing {len(pred_tiffs)} predictions (1st-place chain, config={pp_config})...")
for tif_path in tqdm(pred_tiffs, desc="Postprocessing"):
    raw = tifffile.imread(str(tif_path))
    processed = apply_first_place(raw, pp_config)
    tifffile.imwrite(POSTPROCESSED_DIR / tif_path.name, processed.astype(np.uint8))
print("Postprocessing done ->", POSTPROCESSED_DIR)

## Sanity check: dimensions and dtype

Not part of the source pipeline -- added here because the competition rules are explicit
that each mask "must match the dimensions of the source image exactly, and use the same data
type as the train mask" (uint8). Cheap to check, expensive to get wrong.

In [ ]:
train_labels_dir = INPUT_DIR / "train_labels"
sample_train_label = next(train_labels_dir.glob("*.tif"))
expected_dtype = tifffile.imread(str(sample_train_label)).dtype
print(f"Expected dtype (from a train label): {expected_dtype}")

all_ok = True
for pred_path in sorted(POSTPROCESSED_DIR.glob("*.tif")):
    case_id = pred_path.stem
    src_path = INPUT_DIR / "test_images" / f"{case_id}.tif"
    pred_arr = tifffile.imread(str(pred_path))
    src_arr = tifffile.imread(str(src_path))

    shape_ok = pred_arr.shape == src_arr.shape
    dtype_ok = pred_arr.dtype == expected_dtype
    all_ok &= shape_ok and dtype_ok

    print(f"{case_id}: pred shape={pred_arr.shape} dtype={pred_arr.dtype} | "
          f"src shape={src_arr.shape} | shape_ok={shape_ok} dtype_ok={dtype_ok}")

assert all_ok, "Dimension/dtype mismatch detected -- fix before submitting"
print("\nAll predictions match source dimensions and expected dtype.")


## Generate submission.zip

Verbatim from `baselinerun/src/training/submission.py::generate_submission`.

In [ ]:
def generate_submission(predictions_tiff_dir: Path, output_zip: Path, delete_after_zip: bool = True) -> Optional[Path]:
    tiff_files = sorted(predictions_tiff_dir.glob("*.tif"))
    if not tiff_files:
        print(f"No TIFF files found in {predictions_tiff_dir}")
        return None

    print(f"Creating submission ZIP with {len(tiff_files)} files...")
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
        for tiff_path in tqdm(tiff_files, desc="Zipping predictions"):
            zipf.write(tiff_path, tiff_path.name)
            if delete_after_zip:
                tiff_path.unlink()

    zip_size_mb = output_zip.stat().st_size / (1024 * 1024)
    print(f"Submission saved: {output_zip} ({zip_size_mb:.1f} MB)")
    return output_zip


submission_path = generate_submission(POSTPROCESSED_DIR, SUBMISSION_ZIP)
assert submission_path is not None and submission_path.exists()
print("\nDone:", submission_path)
